<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/WeeklyMonthly_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automation v1 — Interactive Report Visualization Generator

Notebook ini adalah versi **Mode A — Interactive** untuk kebutuhan report **Prabowo, DPR RI, dan Lingkungan**.

### Architecture
`Google Sheets → Data Loader → Validation → Period Resolver → Processing → Visualization Registry → Interactive Control Panel → Output`

### Prinsip utama
- **Tidak ada manual upload file.** Data dibaca langsung dari Google Spreadsheet.
- **Prabowo dan DPR RI memakai schema dan visualization engine yang sama.**
- **Lingkungan memakai schema / processing engine terpisah.**
- Pilihan report: **Daily / Weekly / Monthly**.
- Pilihan visualisasi berubah secara **dynamic** sesuai object dan report type.
- Tersedia **Generate All** atau memilih visualisasi tertentu.
- Ada **Validation Layer** dan **Data Quality Report** sebelum chart dibuat.
- Nama output dibuat otomatis.

> **Sebelum menjalankan notebook:** isi `SPREADSHEET_URL` dan pastikan nama worksheet pada Configuration sesuai Google Spreadsheet Anda.

# 1. Installation

Jalankan sekali setiap runtime Colab baru.

In [ ]:
!pip install -q gspread geopandas SciencePlots ipywidgets

# 2. Imports

In [ ]:
import re                                  # regex, dipakai di slugify() buat bersihin nama file
import math                                # dipakai di plot_sentiment_trend() buat hitung step label sumbu-x
import warnings                            # dipakai buat matiin FutureWarning di baris paling bawah
from pathlib import Path                   # penanganan path folder/file, dipakai di seluruh Output Manager
from datetime import date                  # dipakai di resolve_report_period() dan default value date picker

import numpy as np                         # operasi numerik & np.nan, dipakai luas di processing & chart
import pandas as pd                        # DataFrame, tulang punggung baca/olah/filter data dari sheet
import matplotlib.pyplot as plt            # bikin semua figure & axes di fungsi-fungsi plot_*
import matplotlib as mpl                   # dipakai sekali untuk mpl.patches.Patch (legend custom platform)
import geopandas as gpd                    # baca GeoJSON peta Indonesia & plotting choropleth
import gspread                             # koneksi & baca data dari Google Sheets
import ipywidgets as widgets               # semua komponen UI interaktif (dropdown, checkbox, tombol)
import scienceplots                        # side-effect import, daftarin style "science" ke matplotlib

from IPython.display import display, clear_output  # display buat nampilin DataFrame/panel, clear_output buat refresh output cell
from google.colab import auth, drive               # auth buat login Google, drive buat mount Google Drive
from google.auth import default                    # ambil credential default abis auth.authenticate_user()

warnings.filterwarnings("ignore", category=FutureWarning)  # sembunyiin FutureWarning biar output notebook bersih

# 3. Global Configuration

### Yang wajib Anda ubah
1. `SPREADSHEET_URL`
2. Nama worksheet jika tidak sama dengan default di bawah.

### Expected social schema — Prabowo & DPR RI
Kolom minimum:
- `Time_Label`
- `Sentiment_Positive`
- `Sentiment_Negative`
- `Sentiment_Neutral`

Untuk overview per media, notebook akan memakai kolom seperti:
- `Twitter_Positive`, `Twitter_Negative`, `Twitter_Neutral`
- `Instagram_Positive`, dst.
- `Media Mainstream_Positive`, dst.

### Expected environment schema
Kolom minimum:
- `Time_Label`
- `Provinsi`
- `Isu`

In [ ]:
# ============================================================
# GOOGLE SHEET
# ============================================================

SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/16NCOImkeOfk0bTGxkcZ-V2XSIMAqDwZQX_niLgn9V5k/edit?usp=sharing"

WORKSHEET_MAP = {
    "Prabowo": "Daily - Prabowo",
    "DPR RI": "Daily - DPR RI",
    "Lingkungan": "Rekap Isu Lingkungan",
}

SCHEMA_MAP = {
    "Prabowo": "social",
    "DPR RI": "social",
    "Lingkungan": "environment",
}

# ============================================================
# COLUMN CONFIGURATION
# ============================================================

DATE_COLUMN = "Time_Label"
ENV_PROVINCE_COLUMN = "Provinsi"
ENV_ISSUE_COLUMN = "Isu"

SOCIAL_SENTIMENT_COLUMNS = {
    "positive": "Sentiment_Positive",
    "negative": "Sentiment_Negative",
    "neutral": "Sentiment_Neutral",
}

SENTIMENT_ORDER = ["positive", "negative", "neutral"]

PLATFORM_ORDER = [
    "Media Mainstream",
    "Facebook",
    "Twitter",
    "Youtube",
    "Instagram",
    "Tiktok",
    "Thread",
]

# ============================================================
# VISUAL STYLE
# ============================================================

REPORT_MAX_XLABELS = 10
DEFAULT_DPI = 300
HIGH_DPI = 600

SENTIMENT_COLORS = {
    "positive": "#7AD1FF",
    "negative": "#EE4B2B",
    "neutral": "#818589",
}

PLATFORM_COLORS = {
    "Media Mainstream": "#2ecc71",
    "Facebook": "#2980b9",
    "Twitter": "#5b9bd5",
    "Youtube": "#e74c3c",
    "Instagram": "#9b59b6",
    "Tiktok": "#1abc9c",
    "Thread": "#818589",
}

CMAP_NAME = "YlOrRd"
BASE_MAP_COLOR = "#eeeeee"
EDGE_COLOR = "#444444"
ACTIVE_EDGE_COLOR = "#333333"

TITLE_SIZE = 18
LABEL_SIZE = 13
TICK_SIZE = 12
ANNOT_SIZE = 12

plt.style.use(["science", "no-latex"])

# ============================================================
# OUTPUT
# ============================================================

LOCAL_OUTPUT_ROOT = Path("/content/report_outputs")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/Report Visualization")

LOCAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# ENVIRONMENT GEOJSON
# ============================================================

GEO_URL = (
    "https://raw.githubusercontent.com/"
    "superpikar/indonesia-geojson/master/indonesia-province.json"
)

# ============================================================
# RAW MENTION SCHEMA (data mentah per-postingan) — BARU
# ============================================================

RAW_DATE_COLUMN = "Date"
RAW_SENTIMENT_COLUMN = "Sentiment"
RAW_MEDIA_COLUMN = "Media"

MEDIA_TO_PLATFORM = {
    "Twitter": "Twitter",
    "Youtube": "Youtube",
    "Instagram": "Instagram",
    "Facebook": "Facebook",
    "Tiktok": "Tiktok",
    "Thread": "Thread",
    # nilai "Media" apapun di luar daftar ini (portal berita, dsb.)
    # dianggap "Media Mainstream" — cek catatan soal ini kalau hasilnya meleset.
}


def map_media_to_platform(value):
    return MEDIA_TO_PLATFORM.get(str(value).strip(), "Media Mainstream")

# 4. Environment Mapping & Classification Configuration

In [ ]:
PROVINCE_FIX = {
    'NTT': 'Nusa Tenggara Timur',
    'Ntb': 'Nusa Tenggara Barat',
    'NTB': 'Nusa Tenggara Barat',
    'DKI': 'DKI Jakarta',
    'Jakarta': 'DKI Jakarta',
    'Sumut': 'Sumatera Utara',
    'Sumbar': 'Sumatera Barat',
    'Kaltim': 'Kalimantan Timur',
    'Kalbar': 'Kalimantan Barat',
    'Kalsel': 'Kalimantan Selatan',
    'Kalteng': 'Kalimantan Tengah',
    'Sulsel': 'Sulawesi Selatan',
    'Sultra': 'Sulawesi Tenggara',
    'Sulut': 'Sulawesi Utara',
    'Sulteng': 'Sulawesi Tengah',
    'Sulawesi tengah': 'Sulawesi Tengah',
    'Kepulauan bangka Belitung': 'Kepulauan Bangka Belitung',
    'Kepulauan Bangka belitung': 'Kepulauan Bangka Belitung',
    'Bangka Belitung': 'Kepulauan Bangka Belitung',
    'Kelimantan Selatan': 'Kalimantan Selatan',
    'kalimantan Tengah': 'Kalimantan Tengah',
    'Cirebon': 'Jawa Barat',
    'Subang': 'Jawa Barat',
    'Aceh Tengah': 'Aceh',
    'Sumba Timur': 'Nusa Tenggara Timur',
    'Riau, Sumatera Barat': 'Riau',
    'Kepulauan Maluku': 'Maluku',
    'Yogyakarta': 'DI Yogyakarta',
    'DIY': 'DI Yogyakarta',
    'Sulawesi': 'Sulawesi Tengah',
    'Jawa': 'Jawa Tengah',
    'nan': np.nan,
    'NaN': np.nan,
    'None': np.nan,
    '': np.nan,
}

PULAU_MAP = {
    'Aceh': 'Sumatera',
    'Sumatera Utara': 'Sumatera',
    'Sumatera Barat': 'Sumatera',
    'Riau': 'Sumatera',
    'Kepulauan Riau': 'Sumatera',
    'Jambi': 'Sumatera',
    'Bengkulu': 'Sumatera',
    'Sumatera Selatan': 'Sumatera',
    'Kepulauan Bangka Belitung': 'Sumatera',
    'Lampung': 'Sumatera',

    'Banten': 'Jawa',
    'DKI Jakarta': 'Jawa',
    'Jawa Barat': 'Jawa',
    'Jawa Tengah': 'Jawa',
    'DI Yogyakarta': 'Jawa',
    'Daerah Istimewa Yogyakarta': 'Jawa',
    'Jawa Timur': 'Jawa',

    'Kalimantan Barat': 'Kalimantan',
    'Kalimantan Tengah': 'Kalimantan',
    'Kalimantan Selatan': 'Kalimantan',
    'Kalimantan Timur': 'Kalimantan',
    'Kalimantan Utara': 'Kalimantan',

    'Sulawesi Utara': 'Sulawesi',
    'Gorontalo': 'Sulawesi',
    'Sulawesi Tengah': 'Sulawesi',
    'Sulawesi Barat': 'Sulawesi',
    'Sulawesi Selatan': 'Sulawesi',
    'Sulawesi Tenggara': 'Sulawesi',

    'Bali': 'Bali-Nusa Tenggara',
    'Nusa Tenggara Barat': 'Bali-Nusa Tenggara',
    'Nusa Tenggara Timur': 'Bali-Nusa Tenggara',

    'Maluku': 'Maluku',
    'Maluku Utara': 'Maluku',

    'Papua': 'Papua',
    'Papua Barat': 'Papua',
    'Papua Barat Daya': 'Papua',
    'Papua Tengah': 'Papua',
    'Papua Pegunungan': 'Papua',
    'Papua Selatan': 'Papua',
}

ISLAND_ORDER = [
    'Sumatera',
    'Jawa',
    'Kalimantan',
    'Sulawesi',
    'Bali-Nusa Tenggara',
    'Maluku',
    'Papua',
]

ISSUE_RULES = [
    {'category': 'Tambang ilegal/ekstraktif',
     'keywords': ['tambang', 'pertambangan', 'peti', 'emas', 'batubara',
                  'batu bara', 'nikel', 'mining', 'galian', 'tambang ilegal']},
    {'category': 'Konflik agraria',
     'keywords': ['agraria', 'lahan', 'tanah', 'konflik', 'sengketa',
                  'gusur', 'penggusuran']},
    {'category': 'Deforestasi',
     'keywords': ['hutan', 'deforestasi', 'pembabatan', 'penggundulan',
                  'pembalakan', 'illegal logging']},
    {'category': 'Banjir/longsor',
     'keywords': ['banjir', 'longsor', 'banjir bandang']},
    {'category': 'Pencemaran air/limbah',
     'keywords': ['sungai', 'pencemaran air', 'sumur', 'limbah',
                  'tercemar', 'pencemaran', 'minyak', 'solar',
                  'tumpahan', 'polusi air']},
    {'category': 'Sampah',
     'keywords': ['sampah', 'plastik', 'tpa', 'limbah plastik']},
    {'category': 'Kekeringan/Krisis Air',
     'keywords': ['kemarau panjang', 'krisis air', 'kekeringan']},
    {'category': 'Kebakaran lingkungan',
     'keywords': ['kebakaran', 'karhutla', 'lahan terbakar']},
    {'category': 'Abrasi/erosi',
     'keywords': ['abrasi', 'pantai', 'erosi', 'pesisir']},
    {'category': 'Satwa/ekosistem',
     'keywords': ['satwa', 'habitat', 'ekosistem', 'mangrove',
                  'terumbu', 'biodiversitas', 'konservasi']},
    {'category': 'Polusi udara',
     'keywords': ['udara', 'asap', 'emisi', 'polusi udara',
                  'pencemaran udara']},
]

GEO_FIX = {
    'Di. Aceh': 'Aceh',
    'Nanggroe Aceh Darussalam': 'Aceh',
    'Nangroe Aceh Darussalam': 'Aceh',
    'Aceh': 'Aceh',
    'Probanten': 'Banten',
    'Banten': 'Banten',
    'Dki Jakarta': 'DKI Jakarta',
    'Dki Jakarta Raya': 'DKI Jakarta',
    'Daerah Khusus Ibukota Jakarta': 'DKI Jakarta',
    'Daerah Istimewa Yogyakarta': 'DI Yogyakarta',
    'Di Yogyakarta': 'DI Yogyakarta',
    'Yogyakarta': 'DI Yogyakarta',
    'Bangka Belitung': 'Kepulauan Bangka Belitung',
    'Kepulauan Bangka Belitung': 'Kepulauan Bangka Belitung',
    'Nusatenggara Barat': 'Nusa Tenggara Barat',
    'Nusa Tenggara Barat': 'Nusa Tenggara Barat',
    'Nusatenggara Timur': 'Nusa Tenggara Timur',
    'Nusa Tenggara Timur': 'Nusa Tenggara Timur',
    'Irian Jaya Barat': 'Papua Barat',
    'Irian Jaya Tengah': 'Papua Tengah',
    'Irian Jaya Timur': 'Papua',
    'Papua Barat': 'Papua Barat',
    'Papua Tengah': 'Papua Tengah',
    'Papua': 'Papua',
}

# 5. Google Authentication & Data Connector

Jalankan cell ini lalu ikuti prompt autentikasi Google.

Data menggunakan **lazy loading**:
- worksheet baru dibaca ketika object dipilih / report digenerate;
- dataframe kemudian disimpan pada cache runtime;
- tombol **Reload Data** memaksa pembacaan ulang dari Google Sheets.

In [ ]:
def _sheet_values_to_dataframe(worksheet):
    values = worksheet.get_all_values()

    if not values:
        return pd.DataFrame()

    headers = [str(x).strip() for x in values[0]]
    n_cols = len(headers)
    rows = values[1:]
    normalized_rows = [
        row + [""] * (n_cols - len(row)) if len(row) < n_cols
        else row[:n_cols]
        for row in rows
    ]

    df = pd.DataFrame(normalized_rows, columns=headers)

    df = df.replace("", np.nan)
    df = df.dropna(how="all").reset_index(drop=True)
    return df


def convert_raw_mentions_to_wide(df):
    """Ubah data mentah (1 baris = 1 mention) jadi 1 baris = 1 tanggal,
    dengan kolom Sentiment_Positive/Negative/Neutral dan {Platform}_{Sentiment},
    supaya bentuknya sama seperti worksheet lama (pre-aggregated)."""
    out = df.copy()

    out[RAW_DATE_COLUMN] = pd.to_datetime(out[RAW_DATE_COLUMN], errors="coerce")
    out["Sentiment_Norm"] = (
        out[RAW_SENTIMENT_COLUMN].astype(str).str.strip().str.capitalize()
    )
    out["Platform_Norm"] = out[RAW_MEDIA_COLUMN].apply(map_media_to_platform)
    out[DATE_COLUMN] = out[RAW_DATE_COLUMN].dt.normalize()

    # Total sentimen per hari
    daily_sentiment = (
        out.groupby([DATE_COLUMN, "Sentiment_Norm"])
        .size()
        .unstack("Sentiment_Norm", fill_value=0)
        .reset_index()
    )
    daily_sentiment = daily_sentiment.rename(columns={
        "Positive": "Sentiment_Positive",
        "Negative": "Sentiment_Negative",
        "Neutral": "Sentiment_Neutral",
    })
    for col in ["Sentiment_Positive", "Sentiment_Negative", "Sentiment_Neutral"]:
        if col not in daily_sentiment.columns:
            daily_sentiment[col] = 0

    # Total sentimen per hari per platform
    daily_platform = (
        out.groupby([DATE_COLUMN, "Platform_Norm", "Sentiment_Norm"])
        .size()
        .reset_index(name="Count")
    )
    daily_platform["Column"] = (
        daily_platform["Platform_Norm"] + "_" + daily_platform["Sentiment_Norm"]
    )
    daily_platform_wide = (
        daily_platform
        .pivot_table(index=DATE_COLUMN, columns="Column", values="Count", fill_value=0)
        .reset_index()
    )

    return daily_sentiment.merge(daily_platform_wide, on=DATE_COLUMN, how="outer")

In [ ]:
_gc = None            # gspread client, diisi setelah authenticate_google()
_spreadsheet = None   # objek Spreadsheet, diisi setelah authenticate_google()

_dataset_cache = {}   # cache per-object supaya gak ambil data ke Sheets tiap panggil
_geojson_cache = {"gdf": None}  # cache GeoJSON peta Indonesia


def authenticate_google():
    """Login Google + buka SPREADSHEET_URL sekali di awal runtime."""
    global _gc, _spreadsheet

    auth.authenticate_user()
    creds, _ = default()
    _gc = gspread.authorize(creds)
    print("✓ Google authentication successful")

    _spreadsheet = _gc.open_by_url(SPREADSHEET_URL)
    print(f"✓ Spreadsheet connected: {_spreadsheet.title}")

    return _spreadsheet


def load_dataset(object_name, force_reload=False):
    """Ambil DataFrame untuk satu object (Prabowo/DPR RI/Lingkungan).

    Hasil di-cache di _dataset_cache supaya generate_report() yang
    dipanggil berkali-kali gak perlu hit Google Sheets tiap saat.
    Dipanggil ulang dengan force_reload=True dari reload_dataset().
    """
    if object_name not in WORKSHEET_MAP:
        raise ValueError(f"Unknown object: {object_name}")

    if not force_reload and object_name in _dataset_cache:
        return _dataset_cache[object_name]

    if _spreadsheet is None:
        raise RuntimeError(
            "Belum authenticate. Jalankan authenticate_google() dulu."
        )

    worksheet_name = WORKSHEET_MAP[object_name]
    worksheet = _spreadsheet.worksheet(worksheet_name)
    df = _sheet_values_to_dataframe(worksheet)

    # BARU: kalau worksheet-nya masih format mentah per-mention
    # (ada kolom Sentiment mentah), konversi dulu ke bentuk agregat harian.
    if SCHEMA_MAP[object_name] == "social" and RAW_SENTIMENT_COLUMN in df.columns:
        df = convert_raw_mentions_to_wide(df)

    _dataset_cache[object_name] = df
    return df


def reload_dataset(object_name):
    """Dipanggil tombol 'Reload Data' di Control Panel — paksa ambil ulang dari Sheets."""
    return load_dataset(object_name, force_reload=True)


def load_geojson():
    """Baca GeoJSON peta Indonesia dari GEO_URL, di-cache karena filenya besar."""
    if _geojson_cache["gdf"] is None:
        _geojson_cache["gdf"] = gpd.read_file(GEO_URL)
    return _geojson_cache["gdf"]

### Run Authentication

> Jalankan cell berikut **setelah** `SPREADSHEET_URL` sudah diisi.

In [ ]:
authenticate_google()

✓ Google authentication successful
✓ Spreadsheet connected: report_16834_all_all_20260831_20260906_2767_Prabowo


<Spreadsheet 'report_16834_all_all_20260831_20260906_2767_Prabowo' id:16NCOImkeOfk0bTGxkcZ-V2XSIMAqDwZQX_niLgn9V5k>

# 6. Validation Engine & Data Quality Report

In [ ]:
SOCIAL_REQUIRED_COLUMNS = [
    DATE_COLUMN,
    SOCIAL_SENTIMENT_COLUMNS["positive"],
    SOCIAL_SENTIMENT_COLUMNS["negative"],
    SOCIAL_SENTIMENT_COLUMNS["neutral"],
]

ENVIRONMENT_REQUIRED_COLUMNS = [
    DATE_COLUMN,
    ENV_PROVINCE_COLUMN,
    ENV_ISSUE_COLUMN,
]

SOCIAL_NUMERIC_CANDIDATES = set(SOCIAL_SENTIMENT_COLUMNS.values()) | {
    f"{platform}_{sentiment_name.capitalize()}"
    for platform in PLATFORM_ORDER
    for sentiment_name in SENTIMENT_ORDER
}


def required_columns_for(object_name):
    schema = SCHEMA_MAP[object_name]
    return (
        SOCIAL_REQUIRED_COLUMNS
        if schema == "social"
        else ENVIRONMENT_REQUIRED_COLUMNS
    )


def parse_date_column(df):
    out = df.copy()

    if DATE_COLUMN not in out.columns:
        return out

    out[DATE_COLUMN] = pd.to_datetime(
        out[DATE_COLUMN],
        format="mixed",
        dayfirst=True,
        errors="coerce",
    )
    return out


def coerce_social_numeric(df):
    out = df.copy()

    for col in SOCIAL_NUMERIC_CANDIDATES:
        if col in out.columns:
            cleaned = (
                out[col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .str.strip()
                .replace({"nan": np.nan, "None": np.nan, "": np.nan})
            )
            out[col] = pd.to_numeric(cleaned, errors="coerce")

    return out


def _missing_calendar_dates(df):
    if DATE_COLUMN not in df.columns:
        return []

    valid = (
        pd.to_datetime(df[DATE_COLUMN], errors="coerce")
        .dropna()
        .dt.normalize()
        .drop_duplicates()
        .sort_values()
    )

    if valid.empty:
        return []

    expected = pd.date_range(valid.min(), valid.max(), freq="D")
    actual = set(valid.tolist())
    return [d for d in expected if d not in actual]


def validate_dataset(df, object_name):
    schema = SCHEMA_MAP[object_name]
    required = required_columns_for(object_name)

    missing_required = [
        col for col in required if col not in df.columns
    ]

    errors = []
    warnings_list = []

    if df.empty:
        errors.append("Dataset is empty.")

    if missing_required:
        errors.append(
            "Missing required columns: "
            + ", ".join(missing_required)
        )

    if errors:
        return {
            "status": "ERROR",
            "errors": errors,
            "warnings": warnings_list,
            "rows": len(df),
        }

    parsed = parse_date_column(df)
    invalid_dates = int(parsed[DATE_COLUMN].isna().sum())

    if invalid_dates:
        warnings_list.append(
            f"{invalid_dates:,} row(s) have invalid / missing dates."
        )

    duplicates = int(df.duplicated().sum())
    if duplicates:
        warnings_list.append(
            f"{duplicates:,} exact duplicate row(s) found."
        )

    if schema == "social":
        numeric_df = coerce_social_numeric(parsed)
        invalid_numeric = {}
        for col in SOCIAL_SENTIMENT_COLUMNS.values():
            if col in numeric_df.columns:
                invalid_numeric[col] = int(numeric_df[col].isna().sum())

        problematic = {
            k: v for k, v in invalid_numeric.items() if v > 0
        }
        if problematic:
            warnings_list.append(
                "Missing / invalid sentiment numeric values: "
                + ", ".join(f"{k}={v}" for k, v in problematic.items())
            )

    unmapped_province_rows = None
    if schema == "environment":
        temp = parsed.copy()
        temp[ENV_PROVINCE_COLUMN] = (
            temp[ENV_PROVINCE_COLUMN]
            .astype(str)
            .str.strip()
            .replace(PROVINCE_FIX)
        )
        mapped = temp[ENV_PROVINCE_COLUMN].map(PULAU_MAP)
        unmapped_province_rows = int(mapped.isna().sum())
        if unmapped_province_rows:
            warnings_list.append(
                f"{unmapped_province_rows:,} row(s) have unmapped province/island."
            )

    missing_dates = _missing_calendar_dates(parsed)
    if missing_dates:
        warnings_list.append(
            f"{len(missing_dates):,} calendar date(s) missing "
            f"between dataset min/max date."
        )

    return {
        "status": "READY" if not errors else "ERROR",
        "errors": errors,
        "warnings": warnings_list,
        "rows": len(df),
        "invalid_dates": invalid_dates,
        "duplicates": duplicates,
        "missing_calendar_dates": missing_dates,
        "unmapped_province_rows": unmapped_province_rows,  # BARU
        "parsed_df": parsed,
    }


def print_validation_result(result):
    print("=" * 66)
    print("DATA VALIDATION")
    print("=" * 66)
    print(f"Status : {result['status']}")
    print(f"Rows   : {result.get('rows', 0):,}")

    if result.get("errors"):
        print("\nERRORS")
        for item in result["errors"]:
            print(f"  ✗ {item}")

    if result.get("warnings"):
        print("\nWARNINGS")
        for item in result["warnings"]:
            print(f"  ! {item}")

    if not result.get("errors") and not result.get("warnings"):
        print("\n✓ No structural or basic quality issue detected.")


def data_quality_report(
    full_df,
    period_df,
    object_name,
    report_type,
    start_date,
    end_date,
    validation=None,
):
    if validation is None:
        validation = validate_dataset(full_df, object_name)

    report = {
        "Object": object_name,
        "Worksheet": WORKSHEET_MAP[object_name],
        "Schema": SCHEMA_MAP[object_name],
        "Report Type": report_type.title(),
        "Period": (
            f"{start_date:%d %b %Y} — {end_date:%d %b %Y}"
        ),
        "Rows in database": len(full_df),
        "Rows in selected period": len(period_df),
        "Duplicate rows": validation.get("duplicates", 0),
        "Invalid / missing dates": validation.get("invalid_dates", 0),
        "Missing calendar dates": len(
            validation.get("missing_calendar_dates", [])
        ),
        "Status": validation["status"],
    }

    if SCHEMA_MAP[object_name] == "environment":
        report["Unmapped province rows"] = (
            validation.get("unmapped_province_rows") or 0
        )

    quality_df = pd.DataFrame(
        {"Metric": report.keys(), "Value": report.values()}
    )

    display(quality_df)
    return quality_df, validation

# 7. Report Period Resolver

Aturan:
- **Daily** → hanya reference date.
- **Weekly** → Monday–Sunday yang mengandung reference date.
- **Monthly** → calendar month yang mengandung reference date.
- Backend juga sudah mendukung **Custom** untuk pengembangan selanjutnya.

In [ ]:
def resolve_report_period(
    report_type,
    reference_date=None,
    custom_start=None,
    custom_end=None,
):
    report_type = str(report_type).lower().strip()

    ref = pd.Timestamp(
        reference_date if reference_date is not None else date.today()
    ).normalize()

    if report_type == "daily":
        start = end = ref

    elif report_type == "weekly":
        start = ref - pd.Timedelta(days=ref.weekday())
        end = start + pd.Timedelta(days=6)

    elif report_type == "monthly":
        start = ref.replace(day=1)
        end = start + pd.offsets.MonthEnd(1)

    elif report_type == "custom":
        if custom_start is None or custom_end is None:
            raise ValueError(
                "custom_start dan custom_end wajib untuk Custom."
            )
        start = pd.Timestamp(custom_start).normalize()
        end = pd.Timestamp(custom_end).normalize()

    else:
        raise ValueError(
            "report_type must be Daily, Weekly, Monthly, or Custom."
        )

    if start > end:
        raise ValueError("Start date cannot be after end date.")

    return pd.Timestamp(start), pd.Timestamp(end)


def filter_period(df, start_date, end_date, already_parsed=False):
    temp = df if already_parsed else parse_date_column(df)

    if DATE_COLUMN not in temp.columns:
        raise ValueError(f"Missing date column: {DATE_COLUMN}")

    mask = (
        temp[DATE_COLUMN].dt.normalize().between(
            pd.Timestamp(start_date),
            pd.Timestamp(end_date),
            inclusive="both",
        )
    )

    return (
        temp.loc[mask]
        .sort_values(DATE_COLUMN)
        .reset_index(drop=True)
    )

# 8. Social Data Processing Engine — Shared by Prabowo & DPR RI

In [ ]:
def prepare_social_data(df):
    out = parse_date_column(df)
    out = coerce_social_numeric(out)
    return out.sort_values(DATE_COLUMN).reset_index(drop=True)


def social_sentiment_table(df, prepared=None):
    out = prepared if prepared is not None else prepare_social_data(df)

    rename = {
        SOCIAL_SENTIMENT_COLUMNS[s]: s
        for s in SENTIMENT_ORDER
    }

    return (
        out[[DATE_COLUMN] + list(rename.keys())]
        .rename(columns=rename)
    )


def social_platform_long(df, prepared=None):
    out = prepared if prepared is not None else prepare_social_data(df)
    rows = []

    for platform in PLATFORM_ORDER:
        for sentiment_name in SENTIMENT_ORDER:
            col = f"{platform}_{sentiment_name.capitalize()}"

            if col not in out.columns:
                continue

            temp = out[[DATE_COLUMN, col]].copy()
            temp = temp.rename(columns={col: "Count"})
            temp["Platform"] = platform
            temp["Sentiment"] = sentiment_name
            rows.append(temp)

    if not rows:
        raise ValueError(
            "No platform-sentiment columns found. "
            "Expected columns such as Twitter_Positive, "
            "Instagram_Negative, etc."
        )

    long_df = pd.concat(rows, ignore_index=True)
    long_df["Count"] = pd.to_numeric(
        long_df["Count"], errors="coerce"
    ).fillna(0)

    return long_df


def aggregate_sentiment_daily(df, prepared=None):
    out = prepared if prepared is not None else prepare_social_data(df)

    agg = (
        out.groupby(out[DATE_COLUMN].dt.normalize())
        [list(SOCIAL_SENTIMENT_COLUMNS.values())]
        .sum(min_count=1)
        .reset_index()
    )

    agg = agg.rename(
        columns={
            SOCIAL_SENTIMENT_COLUMNS["positive"]: "Sentiment_Positive",
            SOCIAL_SENTIMENT_COLUMNS["negative"]: "Sentiment_Negative",
            SOCIAL_SENTIMENT_COLUMNS["neutral"]: "Sentiment_Neutral",
        }
    )

    return agg

# 9. Environment Processing Engine

In [ ]:
def classify_issue(text):
    text = str(text).lower()

    for rule in ISSUE_RULES:
        if any(keyword in text for keyword in rule["keywords"]):
            return rule["category"]

    return "Isu lingkungan lain"


def prepare_environment_data(df):
    required = [
        DATE_COLUMN,
        ENV_PROVINCE_COLUMN,
        ENV_ISSUE_COLUMN,
    ]
    missing = [x for x in required if x not in df.columns]
    if missing:
        raise ValueError(
            f"Missing environmental columns: {missing}"
        )

    out = parse_date_column(df)
    out[ENV_PROVINCE_COLUMN] = (
        out[ENV_PROVINCE_COLUMN]
        .astype(str)
        .str.strip()
        .replace(PROVINCE_FIX)
    )

    out["Pulau"] = out[ENV_PROVINCE_COLUMN].map(PULAU_MAP)
    out["Isu_Inti"] = out[ENV_ISSUE_COLUMN].apply(classify_issue)

    return out


def aggregate_environment(df, prepared=None):
    out = prepared if prepared is not None else prepare_environment_data(df)

    agg_province = (
        out.groupby(
            ["Pulau", ENV_PROVINCE_COLUMN],
            observed=True,
        )
        .agg(
            jumlah_kasus=(ENV_ISSUE_COLUMN, "count"),
            isu_inti=(
                "Isu_Inti",
                lambda x: (
                    x.value_counts().idxmax()
                    if len(x.dropna()) else "N/A"
                ),
            ),
        )
        .reset_index()
        .rename(columns={ENV_PROVINCE_COLUMN: "Provinsi"})
    )

    agg_issue = (
        out.groupby("Isu_Inti")
        .size()
        .reset_index(name="jumlah_kasus")
        .sort_values("jumlah_kasus", ascending=False)
        .reset_index(drop=True)
    )

    agg_island = (
        out.groupby("Pulau", observed=True)
        .size()
        .reset_index(name="jumlah_kasus")
    )

    agg_island["Pulau"] = pd.Categorical(
        agg_island["Pulau"],
        categories=ISLAND_ORDER,
        ordered=True,
    )

    agg_island = (
        agg_island
        .sort_values("Pulau")
        .reset_index(drop=True)
    )

    other_issues = out.loc[
        out["Isu_Inti"] == "Isu lingkungan lain",
        [ENV_PROVINCE_COLUMN, "Pulau", ENV_ISSUE_COLUMN],
    ].copy()

    return {
        "data": out,
        "agg_province": agg_province,
        "agg_issue": agg_issue,
        "agg_island": agg_island,
        "other_issues": other_issues,
    }


def find_province_column(gdf):
    possible = [
        "Propinsi", "provinsi", "Provinsi",
        "PROVINSI", "name", "NAME_1",
    ]

    for col in possible:
        if col in gdf.columns:
            return col

    raise ValueError(
        "Province-name column was not found in GeoJSON."
    )


def prepare_map_data(agg_province):
    gdf = load_geojson()
    out = gdf.copy()

    source_col = find_province_column(out)
    out = out.rename(columns={source_col: "Provinsi_Geo"})

    out["Provinsi_Geo"] = (
        out["Provinsi_Geo"]
        .astype(str)
        .str.strip()
        .str.title()
    )
    out["Provinsi"] = out["Provinsi_Geo"].replace(GEO_FIX)
    out["Pulau"] = out["Provinsi"].map(PULAU_MAP)

    merged = out.merge(
        agg_province,
        on=["Provinsi", "Pulau"],
        how="left",
    )

    merged["jumlah_kasus"] = (
        merged["jumlah_kasus"]
        .fillna(0)
        .astype(int)
    )
    merged["isu_inti"] = merged["isu_inti"].fillna(
        "Tidak ada isu dalam dataset"
    )

    return merged

# 10. Visualization Library — Shared Social Charts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def _resolve_social_prepared(data, cache=None):
    prepared = (cache or {}).get("social_prepared")
    if prepared is None:
        prepared = prepare_social_data(data)
    return prepared


def plot_sentiment_trend(
    data,
    object_name,
    start_date,
    end_date,
    cache=None,
):
    prepared = _resolve_social_prepared(data, cache)
    daily = aggregate_sentiment_daily(data, prepared=prepared)

    with plt.style.context("default"):
        fig, ax = plt.subplots(figsize=(20, 4))

        for col, key in zip(
            ["Sentiment_Positive", "Sentiment_Negative", "Sentiment_Neutral"],
            SENTIMENT_ORDER,
        ):
            ax.plot(
                daily[DATE_COLUMN], daily[col],
                label=key.capitalize(), color=SENTIMENT_COLORS[key],
                marker="o", linewidth=2,
            )

        step = max(1, math.ceil(len(daily) / REPORT_MAX_XLABELS))
        ax.set_xticks(daily[DATE_COLUMN][::step])
        ax.set_xticklabels(
            daily[DATE_COLUMN][::step].dt.strftime("%d %b"),
            rotation=45, ha="right",
        )

        ax.set_title(
            f"Sentiment Trend of {object_name}\n"
            f"{start_date:%d %b %Y} — {end_date:%d %b %Y}",
            fontsize=TITLE_SIZE, fontweight="bold",
        )
        ax.set_ylabel("Jumlah Mentions", fontsize=LABEL_SIZE)
        ax.tick_params(axis="both", labelsize=TICK_SIZE)
        ax.legend(frameon=False, fontsize=LABEL_SIZE)
        ax.spines[["top", "right"]].set_visible(False)
        fig.tight_layout()

    return fig


def plot_sentiment_platform_overview(
    data,
    object_name,
    start_date,
    end_date,
    cache=None,
):
    prepared = _resolve_social_prepared(data, cache)
    sentiment_data = social_sentiment_table(data, prepared=prepared)
    platform_data = social_platform_long(data, prepared=prepared)

    sent_colors = SENTIMENT_COLORS
    plat_colors = PLATFORM_COLORS
    robj = object_name
    label_threshold = 5
    period_line = f"{start_date:%d %b %Y} - {end_date:%d %b %Y}"

    with plt.style.context("default"):
        fig = plt.figure(figsize=(38, 6))

        # 5 rows: chart / legend-space / spacer / chart / legend-space
        # left column (ax_tl) spans the full height; right column is split
        # into two stacked mini-charts, each with its own reserved legend row
        # so legends never collide with titles of the chart below.
        gs = fig.add_gridspec(
            5, 2,
            width_ratios=[1.3, 1],
            height_ratios=[1, 0.30, 0.25, 1, 0.05],
            hspace=0.05, wspace=0.22,
            top=0.90, bottom=0.10, left=0.06, right=0.98,
        )

        ax_tl = fig.add_subplot(gs[:, 0])   # left: sentiment share per platform
        ax_tr = fig.add_subplot(gs[0, 1])   # top right: overall sentiment shares
        ax_br = fig.add_subplot(gs[3, 1])   # bottom right: platform contribution share

        # ============================================================
        # LEFT: Sentiment share per Media
        # ============================================================
        pivot = (
            platform_data.groupby(["Platform", "Sentiment"])["Count"]
            .sum()
            .unstack("Sentiment")
            .reindex(PLATFORM_ORDER)
            .dropna(how="all")
        )
        pivot = pivot[[s for s in SENTIMENT_ORDER if s in pivot.columns]].fillna(0)

        row_totals = pivot.sum(axis=1)
        pivot_pct = pivot.div(row_totals.replace(0, np.nan), axis=0) * 100
        pivot_pct = pivot_pct.fillna(0)

        y_tl = np.arange(len(pivot_pct))
        left_tl = np.zeros(len(pivot_pct))

        for sentiment_name in SENTIMENT_ORDER:
            if sentiment_name not in pivot_pct.columns:
                continue
            values = pivot_pct[sentiment_name].values
            ax_tl.barh(
                y_tl, values, left=left_tl, height=0.85,
                color=sent_colors[sentiment_name],
                edgecolor="white", linewidth=0.5,
                label=sentiment_name.capitalize(),
            )
            for i, v in enumerate(values):
                if v >= label_threshold:
                    ax_tl.text(
                        left_tl[i] + v / 2, i, f"{v:.1f}%",
                        ha="center", va="center",
                        color="black", fontsize=20, fontweight="bold",
                    )
            left_tl += values

        ax_tl.set_yticks(y_tl)
        ax_tl.set_yticklabels(pivot_pct.index, fontsize=15)
        ax_tl.set_xlim(0, 100)
        ax_tl.set_xlabel("Mentions (%)", fontsize=18)
        ax_tl.set_title(
            f"Sentiment share per Media of {robj}\n{period_line}",
            fontsize=25, fontweight="bold", pad=5,
        )
        ax_tl.legend(
            loc="upper center", bbox_to_anchor=(0.5, -0.07),
            ncol=3, frameon=False, fontsize=18,
        )
        for spine in ["top", "right", "left", "bottom"]:
            ax_tl.spines[spine].set_visible(False)
        ax_tl.tick_params(left=False, labelsize=14)

        # ============================================================
        # TOP RIGHT: Sentiment Shares overall
        # ============================================================
        sent_totals = sentiment_data[SENTIMENT_ORDER].sum()
        sent_grand_total = sent_totals.sum()
        sent_pct_100 = (
            sent_totals / sent_grand_total * 100
            if sent_grand_total else sent_totals * 0
        )

        legend_handles_tr = [
            plt.Rectangle((0, 0), 1, 1, color=sent_colors[s])
            for s in SENTIMENT_ORDER if s in sent_totals.index
        ]
        legend_labels_tr = [
            f"{s.capitalize()}: {int(sent_totals[s]):,}"
            for s in SENTIMENT_ORDER if s in sent_totals.index
        ]

        left_tr = 0
        for sentiment_name in SENTIMENT_ORDER:
            if sentiment_name not in sent_pct_100.index:
                continue
            value = sent_pct_100[sentiment_name]
            ax_tr.barh(
                0, value, left=left_tr, height=0.2, zorder=3,
                color=sent_colors[sentiment_name],
                edgecolor="white", linewidth=0.5,
            )
            if value >= label_threshold:
                ax_tr.text(
                    left_tr + value / 2, 0, f"{value:.1f}%",
                    ha="center", va="center",
                    color="black", fontsize=20, fontweight="bold",
                )
            left_tr += value

        ax_tr.set_title(
            f"Sentiment Shares of {robj}\n{period_line}",
            fontsize=25, fontweight="bold", pad=8,
        )
        ax_tr.set_xlim(0, 100)
        ax_tr.set_xticks(np.arange(0, 101, 10))
        ax_tr.set_ylim(-0.01, 0.01)
        ax_tr.set_yticks([])
        ax_tr.set_xlabel("Share of Mentions (%)", fontsize=18)
        for spine in ["top", "right", "left", "bottom"]:
            ax_tr.spines[spine].set_visible(False)
        ax_tr.tick_params(axis="x", labelsize=12)
        ax_tr.legend(
            legend_handles_tr, legend_labels_tr,
            loc="upper center", bbox_to_anchor=(0.5, -0.20),
            ncol=3, frameon=False, fontsize=20,
        )
        ax_tr.text(
            100, -0.03, f"Total mentions: {int(sent_grand_total):,}",
            ha="right", va="center",
            fontsize=20, fontweight="bold", color="red",
        )

        # ============================================================
        # BOTTOM RIGHT: Platform Contribution Share
        # ============================================================
        plat_totals = platform_data.groupby("Platform")["Count"].sum()
        plat_totals = plat_totals.reindex(plat_colors.keys()).dropna()
        plat_totals = plat_totals.sort_values(ascending=False)
        plat_grand_total = platform_data["Count"].sum()
        plat_pct_frac = plat_totals / plat_grand_total
        plat_pct_100 = plat_pct_frac * 100

        legend_handles_br = [
            plt.Rectangle((0, 0), 1, 1, color=plat_colors[p])
            for p in plat_pct_frac.index
        ]
        legend_labels_br = [
            f"{p}: {int(plat_totals[p]):,} ({plat_pct_frac[p]*100:.1f}%)"
            for p in plat_totals.index
        ]

        left_br = 0
        for plat in plat_totals.index:
            value = plat_pct_100[plat]
            ax_br.barh(
                0, value, left=left_br, height=0.2, zorder=3,
                color=plat_colors[plat], edgecolor="white", linewidth=0.5,
            )
            if value >= label_threshold:
                ax_br.text(
                    left_br + value / 2, 0, f"{value:.1f}%",
                    ha="center", va="center",
                    color="white", fontsize=15, fontweight="bold",
                )
            left_br += value

        ax_br.set_title(
            f"Platform Contribution Share of {robj}\n{period_line}",
             fontsize=25, fontweight="bold", y=0.5)
        ax_br.set_xlim(0, 100)
        ax_br.set_xticks(np.arange(0, 101, 10))
        ax_br.set_ylim(-0.20, 0.47)
        ax_br.set_yticks([])
        ax_br.set_xlabel("Share of Mentions (%)", fontsize=18)
        for spine in ["top", "right", "left", "bottom"]:
            ax_br.spines[spine].set_visible(False)
        ax_br.tick_params(axis="x", labelsize=12)
        ax_br.legend(
            legend_handles_br, legend_labels_br,
            loc="upper center", bbox_to_anchor=(0.5, -0.55),
            ncol=3, frameon=False, fontsize=20,
        )

    return fig

# 11. Visualization Library — Environment Charts

In [ ]:
def _environment_color_scale(values):
    values = pd.Series(values, dtype=float)

    positive = values[values > 0]

    if positive.empty:
        vmin, vmax = 0, 1
    else:
        vmin = float(positive.min())
        vmax = float(positive.max())
        if vmin == vmax:
            vmax = vmin + 1

    norm = plt.Normalize(vmin, vmax)
    cmap = plt.cm.get_cmap(CMAP_NAME)
    return norm, cmap, vmin, vmax


# BARU: dua helper kecil, biar tiap fungsi plot environment gak nulis
# ulang pola "ambil dari cache kalau ada, kalau enggak hitung sendiri".
def _resolve_environment_aggs(data, cache=None):
    aggs = (cache or {}).get("env_aggs")
    if aggs is None:
        aggs = aggregate_environment(data)
    return aggs


def _resolve_environment_map(aggs, cache=None):
    map_gdf = (cache or {}).get("env_map_gdf")
    if map_gdf is None:
        map_gdf = prepare_map_data(aggs["agg_province"])
    return map_gdf


def plot_national_issue_bar(
    data,
    object_name,
    start_date,
    end_date,
    cache=None,  # BARU
):
    aggs = _resolve_environment_aggs(data, cache)
    plot_data = (
        aggs["agg_issue"]
        .sort_values("jumlah_kasus", ascending=True)
        .copy()
    )

    if plot_data.empty:
        raise ValueError(
            "No environmental issue data in selected period."
        )

    values = plot_data["jumlah_kasus"]
    norm, cmap, _, _ = _environment_color_scale(values)
    colors = cmap(norm(values))

    with plt.style.context("default"):
        fig, ax = plt.subplots(figsize=(10, 6))

        ax.barh(plot_data["Isu_Inti"], values, color=colors)

        offset = max(float(values.max()) * 0.015, 0.5)

        for i, value in enumerate(values):
            ax.text(
                value + offset, i, f"{int(value)}",
                va="center", fontsize=ANNOT_SIZE, fontweight="bold",
            )

        ax.set_title(
            "Distribusi Nasional Isu Lingkungan di Indonesia\n"
            f"{start_date:%d %b %Y} — {end_date:%d %b %Y}",
            fontsize=TITLE_SIZE, fontweight="bold",
        )
        ax.set_xlabel("Jumlah kasus", fontsize=LABEL_SIZE)
        ax.tick_params(axis="y", labelsize=LABEL_SIZE)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlim(0, max(values.max() * 1.15, 1))

        fig.tight_layout()

    return fig


def plot_island_distribution_bar(
    data,
    object_name,
    start_date,
    end_date,
    cache=None,  # BARU
):
    aggs = _resolve_environment_aggs(data, cache)

    plot_data = (
        aggs["agg_island"]
        .dropna(subset=["Pulau"])
        .sort_values("jumlah_kasus", ascending=True)
        .copy()
    )

    if plot_data.empty:
        raise ValueError(
            "No island distribution in selected period."
        )

    values = plot_data["jumlah_kasus"]
    norm, cmap, _, _ = _environment_color_scale(values)
    colors = cmap(norm(values))

    with plt.style.context("default"):
        fig, ax = plt.subplots(figsize=(9, 5.5))

        ax.barh(plot_data["Pulau"].astype(str), values, color=colors)

        offset = max(float(values.max()) * 0.015, 0.3)

        for i, value in enumerate(values):
            ax.text(
                value + offset, i, f"{int(value)}",
                va="center", fontsize=ANNOT_SIZE, fontweight="bold",
            )

        ax.set_title(
            "Distribusi Kasus Isu Lingkungan per Pulau\n"
            f"{start_date:%d %b %Y} — {end_date:%d %b %Y}",
            fontsize=TITLE_SIZE, fontweight="bold",
        )
        ax.set_xlabel("Jumlah kasus", fontsize=LABEL_SIZE)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_xlim(0, max(values.max() * 1.15, 1))

        fig.tight_layout()

    return fig


def plot_indonesia_static(
    data,
    object_name,
    start_date,
    end_date,
    cache=None,  # BARU
):
    aggs = _resolve_environment_aggs(data, cache)
    map_gdf = _resolve_environment_map(aggs, cache)

    active = map_gdf[map_gdf["jumlah_kasus"] > 0].copy()

    norm, cmap, vmin, vmax = _environment_color_scale(
        map_gdf["jumlah_kasus"]
    )

    with plt.style.context("default"):
        fig, ax = plt.subplots(figsize=(15, 8))

        map_gdf.plot(
            ax=ax, color=BASE_MAP_COLOR,
            edgecolor=EDGE_COLOR, linewidth=0.5,
        )

        if not active.empty:
            active.plot(
                ax=ax, column="jumlah_kasus", cmap=CMAP_NAME,
                edgecolor=ACTIVE_EDGE_COLOR, linewidth=0.7,
                legend=True, vmin=vmin, vmax=vmax,
                legend_kwds={"label": "Jumlah kasus/isu", "shrink": 0.55},
            )

            points = active.copy()
            points["point"] = points.geometry.representative_point()

            for _, row in points.iterrows():
                x, y = row["point"].x, row["point"].y
                ax.scatter(
                    x, y, s=260, color="white",
                    edgecolor="black", linewidth=1, zorder=5,
                )
                ax.text(
                    x, y, str(int(row["jumlah_kasus"])),
                    ha="center", va="center",
                    fontsize=8, fontweight="bold", zorder=6,
                )

        ax.set_title(
            "Persebaran Isu Lingkungan di Indonesia\n"
            f"{start_date:%d %b %Y} — {end_date:%d %b %Y}",
            fontsize=TITLE_SIZE, fontweight="bold", pad=12,
        )
        ax.axis("off")
        fig.tight_layout()

    return fig


def plot_environment_overview(
    data,
    object_name,
    start_date,
    end_date,
    cache=None,  # BARU
):
    aggs = _resolve_environment_aggs(data, cache)
    map_gdf = _resolve_environment_map(aggs, cache)

    plot_data = (
        aggs["agg_issue"]
        .query("jumlah_kasus > 0")
        .sort_values("jumlah_kasus", ascending=True)
        .copy()
    )

    norm, cmap, _, _ = _environment_color_scale(map_gdf["jumlah_kasus"])

    fig = plt.figure(figsize=(30, 7))
    gs = fig.add_gridspec(1, 2, width_ratios=[6, 4])

    ax_map = fig.add_subplot(gs[0, 0])
    ax_bar = fig.add_subplot(gs[0, 1])

    zero_mask = map_gdf["jumlah_kasus"] == 0

    map_gdf[zero_mask].plot(
        ax=ax_map, color=BASE_MAP_COLOR,
        edgecolor=EDGE_COLOR, linewidth=0.5,
    )

    map_gdf[~zero_mask].plot(
        ax=ax_map, column="jumlah_kasus", cmap=CMAP_NAME, norm=norm,
        edgecolor=ACTIVE_EDGE_COLOR, linewidth=0.6,
    )

    active = map_gdf.loc[map_gdf["jumlah_kasus"] > 0].copy()

    if not active.empty:
        active["point"] = active.geometry.representative_point()

        for _, row in active.iterrows():
            x, y = row["point"].x, row["point"].y
            ax_map.scatter(
                x, y, s=260, color="white",
                edgecolor="black", linewidth=1, zorder=5,
            )
            ax_map.text(
                x, y, str(int(row["jumlah_kasus"])),
                fontsize=11, ha="center", va="center",
                fontweight="bold", zorder=6,
            )

    ax_map.set_axis_off()
    ax_map.set_title(
        "Persebaran Isu Lingkungan di Indonesia\n"
        f"{start_date:%d %b %Y} — {end_date:%d %b %Y}",
        fontsize=20, fontweight="bold",
    )

    if not plot_data.empty:
        values = plot_data["jumlah_kasus"].values
        colors = cmap(norm(values))
        y = np.arange(len(plot_data))

        ax_bar.barh(y, values, color=colors, edgecolor=EDGE_COLOR, linewidth=0.5)
        ax_bar.set_yticks(y)
        ax_bar.set_yticklabels(
            plot_data["Isu_Inti"], fontsize=16, fontweight="semibold",
        )

        for i, value in enumerate(values):
            ax_bar.text(
                value, y[i], f" {int(value)}",
                va="center", ha="left", fontsize=14, fontweight="semibold",
            )

    ax_bar.set_xlabel("Jumlah Kasus", fontsize=16)
    ax_bar.set_title(
        "Distribusi Nasional Isu Lingkungan di Indonesia\n"
        f"{start_date:%d %b %Y} — {end_date:%d %b %Y}",
        fontsize=18, fontweight="bold",
    )

    for spine in ["top", "right", "left", "bottom"]:
        ax_bar.spines[spine].set_visible(False)

    fig.tight_layout()
    return fig

# 12. Visualization Registry & Report Registry

`VISUALIZATION_REGISTRY` menentukan **fungsi chart**.

`REPORT_REGISTRY` menentukan **chart apa yang muncul di UI** untuk kombinasi object + Daily/Weekly/Monthly.

In [ ]:
VISUALIZATION_REGISTRY = {
    "sentiment_trend": {
        "label": "Sentiment Trend",
        "function": plot_sentiment_trend,
        "schema": "social",
    },
    "sentiment_platform_overview": {
        "label": "Sentiment & Platform Overview",
        "function": plot_sentiment_platform_overview,
        "schema": "social",
    },
    "environment_overview": {
        "label": "Environment Overview (Map + Issues)",
        "function": plot_environment_overview,
        "schema": "environment",
        "needs_map": True,  # BARU — dipakai generate_report buat tau
                             # kapan perlu siapin map_gdf di cache
    },
    "national_issue_distribution": {
        "label": "National Issue Distribution",
        "function": plot_national_issue_bar,
        "schema": "environment",
    },
    "island_distribution": {
        "label": "Island Distribution",
        "function": plot_island_distribution_bar,
        "schema": "environment",
    },
    "indonesia_map": {
        "label": "Indonesia Issue Map",
        "function": plot_indonesia_static,
        "schema": "environment",
        "needs_map": True,
    },
}


REPORT_REGISTRY = {
    "Prabowo": {
        "daily": [
            "sentiment_trend",
            "sentiment_platform_overview",
        ],
        "weekly": [
            "sentiment_trend",
            "sentiment_platform_overview",
        ],
        "monthly": [
            "sentiment_trend",
            "sentiment_platform_overview",
        ],
    },

    "DPR RI": {
        "daily": [
            "sentiment_trend",
            "sentiment_platform_overview",
        ],
        "weekly": [
            "sentiment_trend",
            "sentiment_platform_overview",
        ],
        "monthly": [
            "sentiment_trend",
            "sentiment_platform_overview",
        ],
    },

    "Lingkungan": {
        "daily": [
            "national_issue_distribution",
            "island_distribution",
            "indonesia_map",
        ],
        "weekly": [
            "environment_overview",
            "national_issue_distribution",
            "island_distribution",
            "indonesia_map",
        ],
        "monthly": [
            "environment_overview",
            "national_issue_distribution",
            "island_distribution",
            "indonesia_map",
        ],
    },
}


def available_visualizations(object_name, report_type):
    return REPORT_REGISTRY[
        object_name
    ][report_type.lower()]

# 13. Output Manager & Report Generator

In [ ]:
def slugify(text):
    text = str(text).strip().replace(" ", "_")
    text = re.sub(r"[^A-Za-z0-9_\-]+", "", text)
    return text


def output_filename(
    object_name, report_type, start_date, end_date, visualization_key,
):
    object_slug = slugify(object_name)
    report_slug = slugify(report_type.lower())
    visual_slug = slugify(visualization_key)

    if pd.Timestamp(start_date) == pd.Timestamp(end_date):
        period_slug = pd.Timestamp(start_date).strftime("%Y%m%d")
    else:
        period_slug = (
            f"{pd.Timestamp(start_date):%Y%m%d}-"
            f"{pd.Timestamp(end_date):%Y%m%d}"
        )

    return f"{object_slug}_{report_slug}_{period_slug}_{visual_slug}.png"


def ensure_drive_mounted():
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")


def resolve_output_folder(
    object_name, report_type, start_date, save_to_drive=False,
):
    root = DRIVE_OUTPUT_ROOT if save_to_drive else LOCAL_OUTPUT_ROOT

    if save_to_drive:
        ensure_drive_mounted()

    report_type = report_type.lower()

    if report_type == "daily":
        subpath = Path(
            "Daily", f"{start_date:%Y}", f"{start_date:%m}",
            f"{start_date:%d}", slugify(object_name),
        )
    elif report_type == "weekly":
        iso = start_date.isocalendar()
        subpath = Path(
            "Weekly", f"{start_date:%Y}", f"W{iso.week:02d}",
            slugify(object_name),
        )
    elif report_type == "monthly":
        subpath = Path(
            "Monthly", f"{start_date:%Y}", f"{start_date:%m}",
            slugify(object_name),
        )
    else:
        subpath = Path("Custom", f"{start_date:%Y}", slugify(object_name))

    folder = root / subpath
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def generate_visualization(
    visualization_key,
    period_df,
    object_name,
    report_type,
    start_date,
    end_date,
    save_output=True,
    save_to_drive=False,
    cache=None,  # BARU
):
    if visualization_key not in VISUALIZATION_REGISTRY:
        raise KeyError(f"Unknown visualization: {visualization_key}")

    spec = VISUALIZATION_REGISTRY[visualization_key]

    if spec["schema"] != SCHEMA_MAP[object_name]:
        raise ValueError(
            f"{visualization_key} requires {spec['schema']} schema, "
            f"but {object_name} is {SCHEMA_MAP[object_name]}."
        )

    fig = spec["function"](
        data=period_df,
        object_name=object_name,
        start_date=start_date,
        end_date=end_date,
        cache=cache,  # BARU
    )

    saved_path = None

    if save_output:
        folder = resolve_output_folder(
            object_name=object_name,
            report_type=report_type,
            start_date=start_date,
            save_to_drive=save_to_drive,
        )

        filename = output_filename(
            object_name=object_name,
            report_type=report_type,
            start_date=start_date,
            end_date=end_date,
            visualization_key=visualization_key,
        )

        saved_path = folder / filename
        fig.savefig(saved_path, dpi=DEFAULT_DPI, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    return saved_path


def generate_report(
    object_name,
    report_type,
    reference_date,
    selected_visualizations=None,
    generate_all=False,
    save_output=True,
    save_to_drive=False,
):
    report_type = report_type.lower()

    full_df = load_dataset(object_name)

    validation = validate_dataset(full_df, object_name)
    print_validation_result(validation)

    if validation["status"] == "ERROR":
        raise ValueError(
            "Dataset failed validation. Report generation stopped."
        )

    start_date, end_date = resolve_report_period(
        report_type=report_type,
        reference_date=reference_date,
    )

    # BARU: pakai full_df yang sudah diparse dari validate_dataset,
    # jadi parse_date_column gak jalan ulang di sini (sebelumnya ini
    # yang bikin kolom tanggal full_df diparse sampai 3x per report).
    parsed_full_df = validation.get("parsed_df")
    if parsed_full_df is not None:
        period_df = filter_period(
            parsed_full_df,
            start_date=start_date,
            end_date=end_date,
            already_parsed=True,
        )
    else:
        period_df = filter_period(
            full_df, start_date=start_date, end_date=end_date,
        )

    if period_df.empty:
        raise ValueError(
            "Selected report period contains zero rows. "
            "Choose another reference date or check the database."
        )

    print("\n")
    print("=" * 66)
    print("DATA QUALITY REPORT")
    print("=" * 66)

    # BARU: validation dioper, jadi data_quality_report gak panggil
    # validate_dataset(full_df, ...) dari nol lagi (sebelumnya dobel).
    quality_df, _ = data_quality_report(
        full_df=full_df,
        period_df=period_df,
        object_name=object_name,
        report_type=report_type,
        start_date=start_date,
        end_date=end_date,
        validation=validation,
    )

    allowed = available_visualizations(object_name, report_type)

    if generate_all:
        chosen = allowed
    else:
        chosen = selected_visualizations or []

    chosen = [x for x in chosen if x in allowed]

    if not chosen:
        raise ValueError("No visualization selected.")

    # BARU: cache dibangun sekali di sini, dipakai bareng oleh semua
    # visualisasi yang dipilih dalam satu kali generate report.
    # Ini titik yang paling nendang: untuk Lingkungan "Generate All"
    # (4 chart), classify_issue() yang tadinya jalan 4x sekarang 1x.
    # Untuk Prabowo/DPR RI "Generate All" (2 chart), prepare_social_data
    # yang tadinya jalan sampai 3x sekarang 1x.
    cache = {}
    schema = SCHEMA_MAP[object_name]

    if schema == "social":
        cache["social_prepared"] = prepare_social_data(period_df)

    elif schema == "environment":
        env_prepared = prepare_environment_data(period_df)
        env_aggs = aggregate_environment(period_df, prepared=env_prepared)
        cache["env_prepared"] = env_prepared
        cache["env_aggs"] = env_aggs

        needs_map = any(
            VISUALIZATION_REGISTRY[key].get("needs_map", False)
            for key in chosen
        )
        if needs_map:
            cache["env_map_gdf"] = prepare_map_data(env_aggs["agg_province"])

    print("\n")
    print("=" * 66)
    print("GENERATING VISUALIZATIONS")
    print("=" * 66)

    results = []

    for key in chosen:
        label = VISUALIZATION_REGISTRY[key]["label"]

        print(f"\n→ {label}")

        try:
            saved_path = generate_visualization(
                visualization_key=key,
                period_df=period_df,
                object_name=object_name,
                report_type=report_type,
                start_date=start_date,
                end_date=end_date,
                save_output=save_output,
                save_to_drive=save_to_drive,
                cache=cache,  # BARU
            )

            results.append({
                "Visualization": label,
                "Status": "SUCCESS",
                "Output": (
                    str(saved_path) if saved_path is not None else "Notebook only"
                ),
            })

            print("  ✓ Success")

        except Exception as exc:
            results.append({
                "Visualization": label,
                "Status": "ERROR",
                "Output": str(exc),
            })
            print(f"  ✗ {exc}")

    summary_df = pd.DataFrame(results)

    print("\n")
    print("=" * 66)
    print("REPORT GENERATION SUMMARY")
    print("=" * 66)
    display(summary_df)

    return {
        "full_data": full_df,
        "period_data": period_df,
        "quality": quality_df,
        "summary": summary_df,
        "start_date": start_date,
        "end_date": end_date,
    }

# 14. Interactive Control Panel

### Cara pakai
1. Pilih **Research Object**.
2. Pilih **Daily / Weekly / Monthly**.
3. Pilih **Reference Date**.
4. Visualization checklist akan berubah otomatis.
5. Pilih chart tertentu atau centang **Generate All**.
6. Opsional: centang **Save output** dan **Save to Google Drive**.
7. Klik **Generate Report**.
8. Jika Google Sheet baru saja diperbarui, klik **Reload Data** sebelum generate ulang.

In [ ]:
# ============================================================
# WIDGETS
# ============================================================

object_dropdown = widgets.Dropdown(
    options=list(WORKSHEET_MAP.keys()),
    value="Prabowo",
    description="Object:",
    layout=widgets.Layout(width="420px"),
)

report_dropdown = widgets.Dropdown(
    options=["Daily", "Weekly", "Monthly"],
    value="Weekly",
    description="Report:",
    layout=widgets.Layout(width="420px"),
)

reference_date_picker = widgets.DatePicker(
    description="Date:",
    value=date.today(),
    layout=widgets.Layout(width="420px"),
)

resolved_period_html = widgets.HTML()

generate_all_toggle = widgets.Checkbox(
    value=True,
    description="Generate All",
    indent=False,
)

save_output_toggle = widgets.Checkbox(
    value=True,
    description="Save output",
    indent=False,
)

save_drive_toggle = widgets.Checkbox(
    value=False,
    description="Save to Google Drive",
    indent=False,
)

reload_button = widgets.Button(
    description="Reload Data",
    button_style="warning",
    icon="refresh",
)

generate_button = widgets.Button(
    description="Generate Report",
    button_style="success",
    icon="play",
)

visualization_box = widgets.VBox()
control_output = widgets.Output()

_visualization_checkboxes = {}


# ============================================================
# DYNAMIC UI LOGIC
# ============================================================

def refresh_resolved_period(*args):
    try:
        ref = reference_date_picker.value or date.today()
        start, end = resolve_report_period(
            report_dropdown.value,
            reference_date=ref,
        )

        resolved_period_html.value = (
            "<b>Resolved period:</b> "
            f"{start:%d %b %Y} → {end:%d %b %Y}"
        )

    except Exception as exc:
        resolved_period_html.value = (
            f"<b>Period error:</b> {exc}"
        )


def refresh_visualization_options(*args):
    global _visualization_checkboxes

    object_name = object_dropdown.value
    report_type = report_dropdown.value.lower()

    available = available_visualizations(
        object_name,
        report_type,
    )

    _visualization_checkboxes = {}

    children = []

    for key in available:
        cb = widgets.Checkbox(
            value=True,
            description=VISUALIZATION_REGISTRY[key]["label"],
            indent=False,
            disabled=generate_all_toggle.value,
            layout=widgets.Layout(width="500px"),
        )
        _visualization_checkboxes[key] = cb
        children.append(cb)

    visualization_box.children = tuple(children)
    refresh_resolved_period()


def on_generate_all_change(change):
    for cb in _visualization_checkboxes.values():
        cb.disabled = bool(change["new"])


def selected_visualization_keys():
    return [
        key
        for key, cb in _visualization_checkboxes.items()
        if cb.value
    ]


def on_reload_clicked(button):
    with control_output:
        clear_output(wait=True)

        try:
            obj = object_dropdown.value
            df = reload_dataset(obj)
            validation = validate_dataset(df, obj)

            print(
                f"✓ Reloaded {obj}: {len(df):,} rows"
            )
            print_validation_result(validation)

        except Exception as exc:
            print(f"✗ Reload failed: {exc}")


def on_generate_clicked(button):
    with control_output:
        clear_output(wait=True)

        try:
            ref = reference_date_picker.value

            if ref is None:
                raise ValueError(
                    "Please select a reference date."
                )

            result = generate_report(
                object_name=object_dropdown.value,
                report_type=report_dropdown.value,
                reference_date=ref,
                selected_visualizations=(
                    selected_visualization_keys()
                ),
                generate_all=generate_all_toggle.value,
                save_output=save_output_toggle.value,
                save_to_drive=save_drive_toggle.value,
            )

        except Exception as exc:
            print("=" * 66)
            print("REPORT GENERATION ERROR")
            print("=" * 66)
            print(exc)


object_dropdown.observe(
    refresh_visualization_options,
    names="value",
)
report_dropdown.observe(
    refresh_visualization_options,
    names="value",
)
reference_date_picker.observe(
    refresh_resolved_period,
    names="value",
)
generate_all_toggle.observe(
    on_generate_all_change,
    names="value",
)

reload_button.on_click(on_reload_clicked)
generate_button.on_click(on_generate_clicked)

refresh_visualization_options()
refresh_resolved_period()


# ============================================================
# PANEL LAYOUT
# ============================================================

title = widgets.HTML(
    value=(
        "<h2 style='margin-bottom:4px'>"
        "Report Visualization Generator"
        "</h2>"
        "<p>Mode A — Interactive</p>"
    )
)

panel = widgets.VBox([
    title,
    object_dropdown,
    report_dropdown,
    reference_date_picker,
    resolved_period_html,
    widgets.HTML("<hr><b>Visualizations</b>"),
    generate_all_toggle,
    visualization_box,
    widgets.HTML("<hr><b>Output</b>"),
    save_output_toggle,
    save_drive_toggle,
    widgets.HBox([reload_button, generate_button]),
])

display(panel)
display(control_output)

Output()